# Audit qualité avant validation du nettoyage
Ce notebook ne modifie aucun fichier de données et ne déclenche aucun téléchargement.
Il distingue **signalement**, **erreur confirmée** et **règle analytique à décider**.
Les exemples sont les premières lignes rencontrées, pas un échantillon représentatif.
Les compteurs peuvent se recouper : ne pas les additionner.


In [1]:
from pathlib import Path
import sys, json
import pandas as pd
from IPython.display import display
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "data/cleaned/historical").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Ouvre le notebook dans le projet avec ses données locales.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.analytics.quality import audit_year, FLAGS
HISTORICAL = ROOT / "data/cleaned/historical"


## 1. Bilan des vingt années
Pourcentage = lignes signalées / éléments de prix dans le XML. Les lignes de stations sans prix sont présentées séparément. Une ligne vide `<prix/>` compte comme élément prix et peut porter plusieurs signalements.


In [2]:
reports = [json.loads(p.read_text(encoding="utf-8")) for p in sorted((ROOT / "data/quality").glob("[0-9][0-9][0-9][0-9].json"))]
quality = pd.DataFrame(reports).set_index("year")
counts = quality[["station_count", "source_price_elements", "no_price", *FLAGS]].copy()
percentages = quality[FLAGS].div(quality["source_price_elements"].replace(0, float("nan")), axis=0).mul(100)
display(counts)
display(percentages.round(3).add_suffix(" (%)"))
assert (quality["retained_rows"] == quality["source_price_elements"] + quality["no_price"]).all()
print("Comptages cohérents : éléments prix + stations sans prix = lignes conservées.")


,station_count,source_price_elements,no_price,invalid_price,invalid_date,unknown_fuel,suspect_price,outside_archive_year,duplicate_price,conflicting_price
year,,,,,,,,,,
2007,7904,866993,0,548,548,548,160,0,0,4
2008,8394,1422159,0,605,605,605,50,0,0,6
2009,9387,1398296,3,1229,1229,1229,70,0,0,192
2010,10130,1445254,7,1718,1718,1718,41,0,0,0
2011,10001,1613150,10,1338,1338,1338,33,0,0,2
2012,10256,1904467,9,1229,1229,1229,30,0,0,2
2013,10807,2152004,6,1415,1415,1415,27,0,0,312
2014,11064,2199274,7,1326,1326,1326,0,0,0,14
2015,12333,3307860,14,2601,2601,2601,1,0,6312,2576


,invalid_price (%),invalid_date (%),unknown_fuel (%),suspect_price (%),outside_archive_year (%),duplicate_price (%),conflicting_price (%)
year,,,,,,,
2007,0.063,0.063,0.063,0.018,0.0,0.000,0.000
2008,0.043,0.043,0.043,0.004,0.0,0.000,0.000
2009,0.088,0.088,0.088,0.005,0.0,0.000,0.014
2010,0.119,0.119,0.119,0.003,0.0,0.000,0.000
2011,0.083,0.083,0.083,0.002,0.0,0.000,0.000
2012,0.065,0.065,0.065,0.002,0.0,0.000,0.000
2013,0.066,0.066,0.066,0.001,0.0,0.000,0.014
2014,0.060,0.060,0.060,0.000,0.0,0.000,0.001
2015,0.079,0.079,0.079,0.000,0.0,0.191,0.078


Comptages cohérents : éléments prix + stations sans prix = lignes conservées.


## 2. Audit détaillé d’une année, par carburant
La lecture se fait par lots, sans charger toutes les colonnes ni toutes les années en mémoire.
Tous les carburants de l’année sont examinés, y compris les valeurs inconnues ou absentes.


In [3]:
YEAR = 2022
SAMPLE_SIZE = 10
if YEAR not in quality.index:
    raise ValueError("Année absente des rapports locaux")
audit = audit_year(HISTORICAL / f"{YEAR}.parquet", sample_size=SAMPLE_SIZE)
assert audit["rows"] == quality.loc[YEAR, "retained_rows"]
display(audit["summary"])
print(f"{audit['rows']:,} lignes lues ; {audit['no_price']:,} stations sans élément prix.")


,fuel,declarations,au_moins_un_signalement,invalid_price,invalid_date,unknown_fuel,suspect_price,outside_archive_year,duplicate_price,conflicting_price,invalid_price_pct,invalid_date_pct,unknown_fuel_pct,suspect_price_pct,outside_archive_year_pct,duplicate_price_pct,conflicting_price_pct,au_moins_un_signalement_pct
0,(absent),4010,4010,4010,4010,4010,0,0,0,0,100.0,100.0,100.0,0.000000,0.0,0.0,0.000000,100.000000
1,E10,1247056,380,0,0,0,36,0,0,344,0.0,0.0,0.0,0.002887,0.0,0.0,0.027585,0.030472
2,E85,430263,172,0,0,0,14,0,0,158,0.0,0.0,0.0,0.003254,0.0,0.0,0.036722,0.039976
3,GPLc,206917,4,0,0,0,4,0,0,0,0.0,0.0,0.0,0.001933,0.0,0.0,0.000000,0.001933
4,Gazole,1548593,160,0,0,0,67,0,0,93,0.0,0.0,0.0,0.004327,0.0,0.0,0.006005,0.010332
5,SP95,377007,46,0,0,0,22,0,0,24,0.0,0.0,0.0,0.005835,0.0,0.0,0.006366,0.012201
6,SP98,1234760,171,0,0,0,41,0,0,130,0.0,0.0,0.0,0.003320,0.0,0.0,0.010528,0.013849


5,048,636 lignes lues ; 30 stations sans élément prix.


## 3. Examiner les exemples avant de décider
Choisis un indicateur dans `FLAGS`. Les valeurs brutes permettent de vérifier la transformation. Un prix atypique n’est pas une erreur démontrée. Pour un conflit, charge ensuite toutes les déclarations de la station autour de la date concernée.


In [4]:
FLAG = "conflicting_price"
if FLAG not in FLAGS:
    raise ValueError(f"Choisir parmi {FLAGS}")
display(audit["examples"][FLAG])
# Autres exemples disponibles : audit["examples"]["invalid_date"], etc.


,station_id,fuel,price_raw,date_raw,price_eur_l,price_reported_at,source_archive,source_station_index,source_price_index,conflicting_price
0,1340002,E10,1.642,2022-11-29T00:10:00,1.642,2022-11-29 00:10:00,data/raw/annual/2022.zip,68,309,True
1,1340002,E10,1.66,2022-11-29T00:10:00,1.660,2022-11-29 00:10:00,data/raw/annual/2022.zip,68,310,True
2,1340002,E10,1.561,2022-12-20T00:00:00,1.561,2022-12-20 00:00:00,data/raw/annual/2022.zip,68,323,True
3,1340002,E10,1.572,2022-12-20T00:00:00,1.572,2022-12-20 00:00:00,data/raw/annual/2022.zip,68,324,True
4,2100008,Gazole,1.889,2022-11-25T12:42:00,1.889,2022-11-25 12:42:00,data/raw/annual/2022.zip,152,308,True
5,2100008,Gazole,1.913,2022-11-25T12:42:00,1.913,2022-11-25 12:42:00,data/raw/annual/2022.zip,152,309,True
6,2100008,E10,1.779,2022-11-25T12:42:00,1.779,2022-11-25 12:42:00,data/raw/annual/2022.zip,152,647,True
7,2100008,E10,1.803,2022-11-25T12:42:00,1.803,2022-11-25 12:42:00,data/raw/annual/2022.zip,152,648,True
8,2100008,SP98,1.859,2022-11-25T12:42:00,1.859,2022-11-25 12:42:00,data/raw/annual/2022.zip,152,990,True
9,2100008,SP98,1.883,2022-11-25T12:42:00,1.883,2022-11-25 12:42:00,data/raw/annual/2022.zip,152,991,True


## 4. Identifiants, codes postaux et coordonnées
Chaque station source compte une fois, même si elle a des centaines de prix.
Les coordonnées à zéro sont seulement des cas à examiner. Cet audit ne vérifie pas encore l’appartenance géographique exacte à un territoire : une coordonnée non nulle dans les limites mondiales peut être incorrecte.


In [5]:
station_flags = ["code_postal_a_verifier", "coordonnees_absentes", "coordonnees_zero", "identifiant_absent_ou_repete"]
station_audit = audit["stations"]
display(station_audit[station_flags].sum().rename("Stations à examiner"))
display(station_audit.loc[station_audit[station_flags].any(axis=1)].head(20))


code_postal_a_verifier           0
coordonnees_absentes            69
coordonnees_zero                 6
identifiant_absent_ou_repete     0
Name: Stations à examiner, dtype: Int64

,source_station_index,station_id,postal_code,latitude,longitude,latitude_raw,longitude_raw,code_postal_a_verifier,coordonnees_absentes,coordonnees_zero,identifiant_absent_ou_repete
43,44,1200004,01200,NaN,NaN,,,False,True,False,False
156,157,2100014,02100,NaN,NaN,,,False,True,False,False
396,397,4210001,04210,NaN,NaN,,,False,True,False,False
492,493,6000012,06000,NaN,NaN,,,False,True,False,False
940,941,10000017,10000,NaN,NaN,,,False,True,False,False
1095,1096,11400003,11400,NaN,NaN,,,False,True,False,False
1346,1347,13015012,13015,0.0,0.0,0,0,False,False,True,False
1643,1644,13800008,13800,NaN,NaN,,,False,True,False,False
1832,1833,15000015,15000,NaN,NaN,,,False,True,False,False
1878,1879,15500004,15500,NaN,NaN,,,False,True,False,False


## 5. Vérifier prix brut, unité et date sur une station
Les diviseurs décrivent le traitement actuel, pas une validation indépendante. Pour confirmer l’unité, comparer également les éléments XML originaux et la documentation source, notamment en 2021–2022.


In [6]:
print("Diviseur appliqué :", quality.loc[YEAR, "price_divisor"])
STATION_ID = "1000001"  # Modifiable avec un identifiant trouvé dans les exemples.
station_rows = pd.read_parquet(HISTORICAL / f"{YEAR}.parquet", filters=[("station_id", "==", STATION_ID)])
display(station_rows[["fuel", "price_raw", "price_eur_l", "date_raw", "price_reported_at", "duplicate_price", "conflicting_price"]].head(30))


Diviseur appliqué : 1


,fuel,price_raw,price_eur_l,date_raw,price_reported_at,duplicate_price,conflicting_price
0,Gazole,1.572,1.572,2022-01-03T08:44:18,2022-01-03 08:44:18,False,False
1,Gazole,1.56,1.560,2022-01-05T10:08:11,2022-01-05 10:08:11,False,False
2,Gazole,1.603,1.603,2022-01-10T10:09:08,2022-01-10 10:09:08,False,False
3,Gazole,1.616,1.616,2022-01-17T12:34:42,2022-01-17 12:34:42,False,False
4,Gazole,1.616,1.616,2022-01-17T12:34:44,2022-01-17 12:34:44,False,False
5,Gazole,1.616,1.616,2022-01-17T12:34:45,2022-01-17 12:34:45,False,False
6,Gazole,1.652,1.652,2022-01-21T09:46:30,2022-01-21 09:46:30,False,False
7,Gazole,1.643,1.643,2022-01-25T08:25:26,2022-01-25 08:25:26,False,False
8,Gazole,1.643,1.643,2022-01-28T10:21:42,2022-01-28 10:21:42,False,False
9,Gazole,1.643,1.643,2022-01-28T10:21:43,2022-01-28 10:21:43,False,False


## 6. Ruptures, fermetures et services de cette station
Lecture des JSON de l’archive annuelle, sans considérer automatiquement les interruptions actives à la date de chaque prix. Une fin absente est distinguée d’une fin fournie mais non interprétable. Les contrôles ci-dessous portent uniquement sur la station sélectionnée, pas sur l’ensemble de l’année.


In [7]:
records = []
if not station_rows.empty:
    for column in ["outages_json", "closures_json"]:
        text = station_rows.iloc[0][column]
        for item in json.loads(text) if pd.notna(text) else []:
            records.append({"kind": item["tag"], **item["attributes"]})
    for column in ["services_json", "hours_json", "opening_json"]:
        print(column)
        text = station_rows.iloc[0][column]
        display(json.loads(text) if pd.notna(text) else None)
intervals = pd.DataFrame(records)
if not intervals.empty:
    for raw_column, parsed in [("debut", "start"), ("fin", "end")]:
        values = intervals.get(raw_column, pd.Series(pd.NA, index=intervals.index, dtype="string")).astype("string")
        intervals[parsed + "_absent"] = values.isna() | values.str.strip().eq("")
        intervals[parsed] = pd.to_datetime(values.where(~intervals[parsed + "_absent"]), format="mixed", errors="coerce")
        intervals[parsed + "_invalid"] = ~intervals[parsed + "_absent"] & intervals[parsed].isna()
    try:
        intervals["fin_avant_debut"] = intervals["end"] < intervals["start"]
    except TypeError:
        print("Fuseaux ou types de dates hétérogènes : comparaison à examiner manuellement, sans fuseau inventé.")
    display(intervals)
else:
    print("Aucune interruption enregistrée pour cette station dans cette archive.")


services_json


[{'tag': 'services',
  'attributes': {},
  'text': '\n    ',
  'children': [{'tag': 'service',
    'attributes': {},
    'text': 'Station de gonflage',
    'children': []},
   {'tag': 'service',
    'attributes': {},
    'text': 'Vente de gaz domestique (Butane, Propane)',
    'children': []},
   {'tag': 'service',
    'attributes': {},
    'text': 'DAB (Distributeur automatique de billets)',
    'children': []}]}]

hours_json


[{'tag': 'horaires',
  'attributes': {'automate-24-24': ''},
  'text': '\n    ',
  'children': [{'tag': 'jour',
    'attributes': {'id': '1', 'nom': 'Lundi', 'ferme': ''},
    'text': None,
    'children': []},
   {'tag': 'jour',
    'attributes': {'id': '2', 'nom': 'Mardi', 'ferme': ''},
    'text': None,
    'children': []},
   {'tag': 'jour',
    'attributes': {'id': '3', 'nom': 'Mercredi', 'ferme': ''},
    'text': None,
    'children': []},
   {'tag': 'jour',
    'attributes': {'id': '4', 'nom': 'Jeudi', 'ferme': ''},
    'text': None,
    'children': []},
   {'tag': 'jour',
    'attributes': {'id': '5', 'nom': 'Vendredi', 'ferme': ''},
    'text': None,
    'children': []},
   {'tag': 'jour',
    'attributes': {'id': '6', 'nom': 'Samedi', 'ferme': ''},
    'text': None,
    'children': []},
   {'tag': 'jour',
    'attributes': {'id': '7', 'nom': 'Dimanche', 'ferme': ''},
    'text': None,
    'children': []}]}]

opening_json


None

,kind,id,nom,debut,fin,start_absent,start,start_invalid,end_absent,end,end_invalid,fin_avant_debut
0,rupture,4,GPLc,2017-09-16T09:50:23,,False,2017-09-16 09:50:23,False,True,NaT,False,False
1,rupture,5,E10,2018-12-13T09:49:49,,False,2018-12-13 09:49:49,False,True,NaT,False,False
2,rupture,3,E85,2017-09-16T09:50:23,,False,2017-09-16 09:50:23,False,True,NaT,False,False
3,rupture,6,SP98,2022-10-21T07:51:17,2022-10-21T07:52:49,False,2022-10-21 07:51:17,False,False,2022-10-21 07:52:49,False,False
4,rupture,2,SP95,2022-10-11T08:31:05,2022-10-11T17:10:56,False,2022-10-11 08:31:05,False,False,2022-10-11 17:10:56,False,False
5,rupture,2,SP95,2022-10-10T08:45:24,2022-10-11T07:33:21,False,2022-10-10 08:45:24,False,False,2022-10-11 07:33:21,False,False
6,rupture,2,SP95,2022-10-27T09:28:54,2022-11-02T07:52:47,False,2022-10-27 09:28:54,False,False,2022-11-02 07:52:47,False,False
7,rupture,6,SP98,2022-11-17T14:56:40,2022-11-21T08:43:51,False,2022-11-17 14:56:40,False,False,2022-11-21 08:43:51,False,False
8,rupture,2,SP95,2022-11-17T14:56:40,2022-11-21T08:43:51,False,2022-11-17 14:56:40,False,False,2022-11-21 08:43:51,False,False
9,rupture,1,Gazole,2022-10-11T08:31:04,2022-10-11T17:10:55,False,2022-10-11 08:31:04,False,False,2022-10-11 17:10:55,False,False


## 7. Recouvrements entre années — facultatif, une station
Ce contrôle local recherche des observations typées identiques présentes dans deux archives. Il ne constitue pas un audit exhaustif de toutes les stations. Les lignes invalides sont exclues de la comparaison pour éviter de traiter deux valeurs manquantes comme une déclaration identique.


In [8]:
CHECK_CROSS_YEAR = False
YEAR_A, YEAR_B = 2021, 2022
if CHECK_CROSS_YEAR:
    keys = ["station_id", "fuel", "price_reported_at", "price_eur_l"]
    parts = []
    for year in [YEAR_A, YEAR_B]:
        part = pd.read_parquet(HISTORICAL / f"{year}.parquet", columns=keys + ["archive_year"], filters=[("station_id", "==", STATION_ID)])
        parts.append(part.dropna(subset=keys).drop_duplicates(keys))
    overlap = parts[0].merge(parts[1], on=keys, suffixes=("_A", "_B"), validate="one_to_one")
    display(overlap)
    print("Déclarations communes pour cette station :", len(overlap))


## 8. Règles à valider ensemble
Les propositions ci-dessous ne sont pas appliquées par ce notebook. Les exclusions actuellement utilisées dans le dashboard restent des règles d’analyse séparées du stockage complet.


In [9]:
decisions = pd.DataFrame([
    ["Unités", "Confirmer les conversions avec les sources", "À valider"],
    ["Coordonnées", "Examiner les points suspects ; corriger seulement avec preuve", "À valider"],
    ["Doublons exacts", "Conserver la source ; une seule occurrence dans une vue analytique", "À valider"],
    ["Prix contradictoires", "Ne pas choisir arbitrairement un prix", "À valider"],
    ["Prix atypiques", "Examiner les exemples avant exclusion", "À valider"],
    ["Dates hors année", "Examiner la provenance et le recouvrement temporel", "À valider"],
    ["Interruptions", "Distinguer fin absente, invalide et antérieure au début", "À valider"],
], columns=["Sujet", "Proposition", "Décision"])
display(decisions)


,Sujet,Proposition,Décision
0,Unités,Confirmer les conversions avec les sources,À valider
1,Coordonnées,Examiner les points suspects ; corriger seulem...,À valider
2,Doublons exacts,Conserver la source ; une seule occurrence dan...,À valider
3,Prix contradictoires,Ne pas choisir arbitrairement un prix,À valider
4,Prix atypiques,Examiner les exemples avant exclusion,À valider
5,Dates hors année,Examiner la provenance et le recouvrement temp...,À valider
6,Interruptions,"Distinguer fin absente, invalide et antérieure...",À valider


## Tes conclusions
Pour chaque règle : quels exemples as-tu examinés ? Quelle décision retiens-tu ? Pourquoi ? Quel effet sur les observations utilisées dans l’analyse ?


In [10]:
# Tes notes et analyses complémentaires ici.
